In [2]:
import sys
print(sys.executable)

c:\Users\amrit\AppData\Local\Programs\Python\Python314\python.exe


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
print("Imports working 🚀")

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings,ChatHuggingFace,HuggingFaceEndpoint
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv
load_dotenv()

ModuleNotFoundError: No module named 'langchain_huggingface'

In [2]:
@tool
def conversion_factor(base_curr:str,target_curr:str)->float:
    """
    This function fetches the currency conversion rate at real time between base currency and target currency
    """
    url=f'https://v6.exchangerate-api.com/v6/e95ce53494bf36351a73e88c/pair/{base_curr}/{target_curr}'
    response=requests.get(url)

    return response.json()

@tool
def converter(base_curr_val:int,conversion_rate:float)->float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_curr_val*conversion_rate

In [3]:
converter.args

{'base_curr_val': {'title': 'Base Curr Val', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [4]:
conversion_factor.invoke({'base_curr':'USD','target_curr':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1777680001,
 'time_last_update_utc': 'Sat, 02 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1777766401,
 'time_next_update_utc': 'Sun, 03 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.0415}

In [ ]:
model = genai.GenerativeModel("gemini-1.5-flash")

In [14]:
# tools = [conversion_factor, converter]

model_with_tools = model.bind_tools([conversion_factor, converter])

In [15]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [16]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [18]:
ai_messages=model_with_tools.invoke(messages)

In [20]:
ai_messages

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_curr":"INR","target_curr":"USD"}', 'name': 'conversion_factor', 'description': None}, 'id': '78f3aefd9', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 200, 'total_tokens': 262}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de63e-eeda-7b41-b6ee-c62623f3fcf7-0', tool_calls=[{'name': 'conversion_factor', 'args': {'base_curr': 'INR', 'target_curr': 'USD'}, 'id': '78f3aefd9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 62, 'total_tokens': 262})

In [21]:
ai_messages.tool_calls

[{'name': 'conversion_factor',
  'args': {'base_curr': 'INR', 'target_curr': 'USD'},
  'id': '78f3aefd9',
  'type': 'tool_call'}]

In [11]:
messages.append(ai_messages)

In [12]:
ai_messages.tool_calls

[{'name': 'conversion_factor',
  'args': {'base_curr': 'INR', 'target_curr': 'USD'},
  'id': '114ff004c',
  'type': 'tool_call'}]